# All-in-One Vision, Embeddings & Reranker Server for Colab

Provides:
1. `/v1/chat/completions`: Qwen2.5-VL-7B-Instruct for visual document parsing
2. `/v1/embeddings`: BAAI/bge-m3 (1024-dim) for text chunks
3. `/v1/figure-embeddings`: Google SigLIP (1152-dim) for technical diagrams
4. `/v1/rerank`: BAAI/bge-reranker-v2-m3 for high-precision Cross-Encoder reranking

Exposed securely over a free Cloudflare tunnel.

### Step 1: Install Dependencies & Cloudflared

In [ ]:
# Install extra packages without touching Colab pre-installed CUDA PyTorch
!pip install -q transformers==4.49.0 accelerate bitsandbytes qwen-vl-utils sentence-transformers fastapi uvicorn pydantic pillow
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Make sure T4 GPU is selected in Runtime -> Change runtime type")
print("Dependencies installed successfully!")

### Step 2: Load Models (BGE-M3 + SigLIP + Qwen2.5-VL + BGE-Reranker)

In [ ]:
import io
import base64
import torch
from PIL import Image
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")

# 1. Load BGE-M3 Text Embeddings Model (1024 dimensions)
print("Loading BAAI/bge-m3 text embedding model...")
bge_model = SentenceTransformer("BAAI/bge-m3", device=device)
print(f"BGE-M3 loaded successfully on {device}!")

# 2. Load Google SigLIP Vision Embeddings Model (1152 dimensions)
print("Loading google/siglip-so400m-patch14-384 visual model...")
siglip_model_id = "google/siglip-so400m-patch14-384"
siglip_processor = AutoProcessor.from_pretrained(siglip_model_id)
siglip_model = AutoModel.from_pretrained(
    siglip_model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device).eval()
print(f"SigLIP loaded successfully on {device}!")

# 3. Load BGE-Reranker-v2-m3 Cross-Encoder
print("Loading BAAI/bge-reranker-v2-m3 cross-encoder...")
reranker_model = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device)
print(f"BGE-Reranker loaded successfully on {device}!")

# 4. Load Qwen2.5-VL-7B (if on GPU)
vlm_model = None
vlm_processor = None
if device == "cuda":
    vlm_id = "Qwen/Qwen2.5-VL-7B-Instruct"
    print(f"Loading {vlm_id} in 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    vlm_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        vlm_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    vlm_processor = AutoProcessor.from_pretrained(vlm_id, max_pixels=768*768)
    print("Qwen2.5-VL loaded successfully on GPU!")
else:
    print("Skipping VLM on CPU (embeddings are ready).")

print("\nAll requested models are ready!")

### Step 3: Start Cloudflare Tunnel & FastAPI Server

In [ ]:
import os
import re
import time
import subprocess
import traceback
import torch
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from qwen_vl_utils import process_vision_info

if torch.cuda.is_available():
    torch.cuda.empty_cache()
subprocess.run(["pkill", "-f", "cloudflared"])
time.sleep(1)

log_file = "/tmp/cloudflared.log"
if os.path.exists(log_file):
    os.remove(log_file)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--logfile", log_file]
)

tunnel_url = None
print("Waiting for Cloudflare Tunnel URL...")
for _ in range(30):
    time.sleep(1)
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    print("\n=======================================================")
    print(f"SUCCESS! Your public OpenAI endpoint is:\n{tunnel_url}/v1")
    print("\nPaste this into your local .env file:")
    print(f"LM_STUDIO_URL={tunnel_url}/v1")
    print("=======================================================\n")
else:
    print("Could not automatically detect tunnel URL. Check tunnel logs:")
    if os.path.exists(log_file):
        with open(log_file, "r") as f:
            print(f.read())

app = FastAPI()

@app.get("/v1/models")
async def get_models():
    return {
        "data": [
            {"id": "Qwen2.5-VL-7B-Instruct", "object": "model", "owned_by": "qwen"},
            {"id": "BAAI/bge-m3", "object": "model", "owned_by": "baai"},
            {"id": "google/siglip-so400m-patch14-384", "object": "model", "owned_by": "google"},
            {"id": "BAAI/bge-reranker-v2-m3", "object": "model", "owned_by": "baai"}
        ]
    }

# 1. Text Embeddings Endpoint (BGE-M3, 1024-dim)
@app.post("/v1/embeddings")
async def create_embeddings(req: Request):
    try:
        body = await req.json()
        input_data = body.get("input", [])
        if isinstance(input_data, str):
            input_data = [input_data]
        
        vectors = bge_model.encode(
            input_data,
            normalize_embeddings=True,
            batch_size=32,
            show_progress_bar=False
        ).tolist()
        
        data = [
            {"object": "embedding", "index": i, "embedding": vec}
            for i, vec in enumerate(vectors)
        ]
        return {
            "object": "list",
            "data": data,
            "model": "BAAI/bge-m3",
            "usage": {"prompt_tokens": len(input_data) * 25, "total_tokens": len(input_data) * 25}
        }
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": str(e)})

# 2. Cross-Encoder Reranker Endpoint (BGE-Reranker-v2-m3)
@app.post("/v1/rerank")
async def create_rerank(req: Request):
    try:
        body = await req.json()
        query = body.get("query", "")
        documents = body.get("documents", [])
        top_n = body.get("top_n", len(documents))
        
        pairs = [[query, doc if isinstance(doc, str) else doc.get("text", "")] for doc in documents]
        scores = reranker_model.predict(pairs).tolist()
        
        results = [
            {"index": i, "relevance_score": float(score)}
            for i, score in enumerate(scores)
        ]
        results.sort(key=lambda x: x["relevance_score"], reverse=True)
        return {
            "model": "BAAI/bge-reranker-v2-m3",
            "results": results[:top_n]
        }
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": str(e)})

# 3. Figure Visual Embeddings Endpoint (SigLIP, 1152-dim)
@app.post("/v1/figure-embeddings")
async def create_figure_embeddings(req: Request):
    try:
        body = await req.json()
        images_b64 = body.get("images", [])
        pil_images = []
        for b64 in images_b64:
            if "," in b64:
                b64 = b64.split(",", 1)[1]
            img_bytes = base64.b64decode(b64)
            pil_images.append(Image.open(io.BytesIO(img_bytes)).convert("RGB"))
        
        inputs = siglip_processor(images=pil_images, return_tensors="pt").to(device)
        with torch.no_grad():
            feats = siglip_model.get_image_features(**inputs)
            feats = torch.nn.functional.normalize(feats, p=2, dim=1)
        vectors = feats.cpu().float().tolist()
        
        data = [
            {"object": "embedding", "index": i, "embedding": vec}
            for i, vec in enumerate(vectors)
        ]
        return {
            "object": "list",
            "data": data,
            "model": "google/siglip-so400m-patch14-384"
        }
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": str(e)})

# 4. VLM Completions Endpoint (Qwen2.5-VL)
@app.post("/v1/chat/completions")
async def chat_completions(req: Request):
    if vlm_model is None or vlm_processor is None:
        return JSONResponse(status_code=400, content={"error": "VLM not loaded on this instance"})
    try:
        body = await req.json()
        messages = body.get("messages", [])
        max_tokens = min(body.get("max_tokens", 512), 1024)
        temperature = body.get("temperature", 0.1)

        norm_messages = []
        for msg in messages:
            content = msg.get("content")
            if isinstance(content, list):
                norm_content = []
                for part in content:
                    if part.get("type") == "image_url":
                        img_val = part.get("image_url")
                        url = img_val.get("url") if isinstance(img_val, dict) else img_val
                        norm_content.append({"type": "image", "image": url})
                    elif part.get("type") in ["text", "image"]:
                        norm_content.append(part)
                    else:
                        norm_content.append(part)
                norm_messages.append({"role": msg.get("role", "user"), "content": norm_content})
            else:
                norm_messages.append(msg)

        text = vlm_processor.apply_chat_template(
            norm_messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(norm_messages)
        inputs = vlm_processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            generated_ids = vlm_model.generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=(temperature > 0),
                temperature=max(0.01, temperature)
            )
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = vlm_processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": "Qwen2.5-VL-7B-Instruct",
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": output_text},
                "finish_reason": "stop"
            }],
            "usage": {
                "prompt_tokens": len(inputs.input_ids[0]),
                "completion_tokens": len(generated_ids_trimmed[0]),
                "total_tokens": len(inputs.input_ids[0]) + len(generated_ids_trimmed[0])
            }
        }
    except Exception as e:
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": str(e), "traceback": traceback.format_exc()})
    finally:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Run FastAPI server
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()
